In [ ]:
path_to_dodem = '/Users/jmdunca2/do-dem/'
from sys import path as sys_path
sys_path.append(path_to_dodem+'/dodem/')

import HARP_and_age as haa
import nustar_utilities as nuutil
import lightcurves as lc
import visualize_dem_results as viz

import pickle
import importlib

import numpy as np
from matplotlib import pyplot as plt
from astropy import units as u

from astropy.visualization import quantity_support
quantity_support()

import cashstatistic

targets_file='/Users/jmdunca2/do-dem/reference_files/all_targets_postghost_postshut.pickle'

with open(targets_file, 'rb') as f:
    all_targets = pickle.load(f)

QUIESCENCE

For each contiguous group of "quiescent" time intervals, we want to examine the range of NuSTAR (and AIA?) values included, and determine if they include any statistically significant variation. 

One way to look at this would be to use the DEM inputs themselves – see if there is variation between the inputs for different times that is statistically significant. 

Another way would be to make lightcurves for the region for that range of time intervals and look for finer-grained evidence of transient behavior. 

To do this, we would take each time interval (which may be one or more DEM time intervals) and retrieve the .evt file (sunpos) associated. We would also need an appropriate region file – method for that might differ by the region method (fit/double/input). 

We do the selection for time, space, energy for each contiguous interval and save a lightcurve file (dictionary with lightcurves, livetime, etc. plus some other stats).


Then, we plot these lightcurves and institute various tests to evaulate whether they are quiescent. A CSTAT-based test is used to make final choices re quiescence. 

The expected input target file for this analysis is one that has been made with only in initial flare lists. After this analysis, we added two more flare lists: 

- list4 (obvious small transients seen in cases that failed the CSTAT test)
- cstat_elim ("flares" that are just whole intervals that we are rejecting (a convenience method).

The process was:
1. run CSTAT analysis on original target file
2. for bad cases, manually examine for clear transients, manually add to list4
3. re-run preparation of targets file (add_result_params) to remove intervals intersecting with list4 transients
4. re-run CSTAT analysis
5. for bad cases, make new flare list eliminating all time intervals therewithin (idea is to reject non-transient departures from Poisson-like rate – e.g. steady rise/decay, general elevated variability).
6. re-run preparation of targets file (add_result_params) to remove intervals intersecting with cstat_elim transients
7. continue with other analysis using new, confirmed quiescent sample.

In [ ]:
def get_same_region_file_lists(samesames, all_targets, get_agelists=False, noxrt=False, aiaxrt=False):

    #For the list of lists of region names, make a file list for every separate region
    filelist = []
    if get_agelists:
        agelist=[]
    for s in samesames:
        ss_filelist=[]
        if get_agelists:
            ss_agelist=[]
        #For each key-region combo in the list of instances of this region
        for ss in s:
            #sub-keys in the key
            sks = all_targets[ss.split(' ')[0]]['sub_keys']
            #for each sub-key
            for i in range(0, len(sks)):
                #if the sub-key is the key-region combo we want...
                if sks[i] == ss:
                    #print(ss.split(' ')[0], i)
                    if noxrt:
                        ss_filelist.extend(all_targets[ss.split(' ')[0]]['res_file_dict(s)'][i]['quiet files no_xrt'])
                    elif aiaxrt:
                        ss_filelist.extend(all_targets[ss.split(' ')[0]]['res_file_dict(s)'][i]['quiet files aiaxrt'])
                    else:
                        if len(all_targets[ss.split(' ')[0]]['res_file_dict(s)'][i]['quiet files all-inst-gr-corr']) > 0:
                            ss_filelist.extend(all_targets[ss.split(' ')[0]]['res_file_dict(s)'][i]['quiet files all-inst-gr-corr'])
                        else:
                            ss_filelist.extend(all_targets[ss.split(' ')[0]]['res_file_dict(s)'][i]['quiet files all-inst'])

                        #print(ss)
                        #print(ss_filelist)
                    if get_agelists:
                        #print(ss.split(' ')[0], i)
                        #print(ss, all_targets[ss.split(' ')[0]]['Region Ages'])
                        ss_agelist.append(all_targets[ss.split(' ')[0]]['Region Ages'][i])
                        
        filelist.append(ss_filelist)
        if get_agelists:
            agelist.append(ss_agelist)

    if get_agelists:
        return filelist, agelist
    else:
        return filelist


def make_labels(regids):
    

    arlabs=[]
    for r in regids:
        ss = r[0]
        ars = all_targets[ss.split(' ')[0]]['NOAA_ARID']
        locs = all_targets[ss.split(' ')[0]]['loc']
        if len(locs) == 1:
            arlabs.append(all_targets[ss.split(' ')[0]]['NOAA_ARID'][0])
        else:
            if ss[-1]=='0':
                arlabs.append(all_targets[ss.split(' ')[0]]['NOAA_ARID'][0])
            elif ss[-1]=='1':
                arlabs.append(all_targets[ss.split(' ')[0]]['NOAA_ARID'][1])

    #Making labels
    labs=[]
    #For each region
    for r in regids:
        #print(r)
        lab=''
        first=0
        # if 1 == 2:
        #     numstr = ' '
        # else:
        numstr = '' #' #'+str(int(r[0][-1])+1)
        #For each day it was observed
        for ss in r:
            #If it's the first day...
            if first == 0:
                #If the first character is 0 (aka day of month < 10)
                if ss[0] == '0':
                    #'#'+str(int(ss[-1])+1)+
                    lab=lab+ss[1:9]+numstr+' '
                else: 
                    #
                    lab=lab+ss[0:9]+numstr+' '
                first=1
                #print(lab)
            else:
                if ss[0] == '0':
                    lab=ss[1:2]+','+lab
                else:    
                    lab=ss[0:2]+','+lab
    
        labs.append(lab)
    
    plotlabs = [labs[i]+arlabs[i] for i in range(0, len(labs))]

    return plotlabs



def check_contig(time_intervals):
    """
    Return true if the list of start,stop times is contiguous, False if not 
    (plus lists of intervals, indices).

    Contiguous is False if there are no input time intervals. 
    """
    if len(time_intervals) == 0:
        return False, [], []
    #Check if the quiescent times are contiguous for this region. 
    contiguous=True
    #gaps=[]
    intervals=[]
    int_inds=[]
    starter=0
    #For each entry in the list of time intervals
    for i in range(0,len(time_intervals)):
        #print(i, time_intervals[i])
        if not i == len(time_intervals)-1:
            #If the end of this interval doesn't match the beginning of the next.
            if not time_intervals[i][1]==time_intervals[i+1][0]:
                #print('GAP!', times[i:i+3])
                contiguous=False
                #gaps.append(time_intervals[i:i+2])
                intervals.append(time_intervals[starter:i+1])
                int_inds.append((starter, i+1))
                starter=i+1
                #If the last time interval is a separate contiguous interval
                if i+1==len(time_intervals)-1:
                    intervals.append([time_intervals[i+1]])
                    int_inds.append((starter,len(time_intervals)))
        else:
            #if the last time interval is an additional part of an existing contiguous interval
            if time_intervals[i][0] == time_intervals[i-1][1]:
                intervals.append(time_intervals[starter:i+1])
                int_inds.append((starter, i+1))
    
    if contiguous:
        #If there is only one interval making up the contiguous interval
        if len(int_inds) == 0:
           intervals.append(time_intervals)
           int_inds.append((starter, len(time_intervals)))
    
    return contiguous, intervals, int_inds




default_kwargs = {
        'linestyle': 'dashed',
        'linewidth': 1,
        'marker': 'o',
        'markersize': 2,
    }
import matplotlib.dates as mdates


def check_qui(rate, method='adjacent'):
    """
    identify indices where the rate changes outside counting statistics
    
    """
    #print(rate)
    uprate = rate + rate**(1/2)
    downrate = rate - rate**(1/2)
    
    if method == 'adjacent':
        cons=[]
        for r in range(0, len(rate)-1):
            if not (max(downrate[r:r+2]) <= min(uprate[r:r+2])):
                if rate[r+1] > rate[r]:
                    cons.append(1)
                if rate[r] >= rate[r+1]:
                    cons.append(-1)
            else:
                cons.append(0)
            #cons.append(max(downrate[r:r+2]) <= min(uprate[r:r+2]))
        return cons

    if method=='global':
        #global consistency
        GLCONS=True
        baddies=[]
        avgrate=np.sum(rate)/len(rate)
        upavg, downavg = avgrate + 3*np.std(rate), avgrate-3*np.std(rate)
        #print('Average, bounds: ', avgrate, upavg, downavg)
        for r in range(0, len(rate)):
            if not downavg < rate[r] < upavg:
                baddies.append(r)
                GLCONS=False
            #if not (max(downrate[r], downavg) <= min(uprate[r], upavg)):
                if not rate[r] == 0.:
                    baddies.append(r)
                    GLCONS=False
            # if rate[r] > 0:
            #     for r2 in range(0, len(rate)):
            #         if not (max(downrate[r], downrate[r2]) <= min(uprate[r], uprate[r2])):
            #             #print('NOT GLOBALLY CONSISTENT!')
            #             GLCONS=False
        return GLCONS, baddies
        

def plot_nustar_lightcurves_err(save_dir='./', timerange=[], res_save_add='',
                            eranges = [[2.,4.],[4.,6.],[6.,10.]],
                           nukeyword=''):

    """
    To be run after prepare_nustar_lightcurves has already been run for each energy range for the obsid in question.
    
    Single orbit/single obsid functionality.
    
    
    """
    ecolors=['red', 'red', 'blue', 'blue', 'green', 'green',]
    #fig, (ax2, ax3, ax4) = plt.subplots(3, 1, figsize=(15, 12))
    fig, (ax2, ax3) = plt.subplots(2, 1, figsize=(15, 9))
    instrument='NuSTAR'
    clrs=lc.make_colors(26)
    ind=8
    for er in eranges:
        erange=er
                
        data = lc.load_lightcurves(instrument, erange=erange, lc_dir=save_dir, nukeyword=nukeyword)
        
        if data is None:
            print('Missing prepared data - exiting.')
            return


        #Remove times from outside the time range
        times_convertedA = data['FPMA_times']
        times_convertedB = data['FPMB_times']
        time_indsA = np.where(np.logical_and(times_convertedA > tr[0], times_convertedA < tr[1]))[0]
        time_indsB = np.where(np.logical_and(times_convertedA > tr[0], times_convertedA < tr[1]))[0]
        
        times_convertedA=times_convertedA[time_indsA]
        times_convertedB=times_convertedB[time_indsB]
        countrateA = data['FPMA_countrate'][time_indsA]
        lvtA = data['FPMA_livetime'][time_indsA]
        countrateB = data['FPMB_countrate'][time_indsB]
        lvtB = data['FPMB_livetime'][time_indsB]

        #Remove times with zero livetimes (SAA, etc)
        noliveA = np.where(lvtA != 0.)[0]
        noliveB = np.where(lvtB != 0.)[0]
        times_convertedA=times_convertedA[noliveA]
        times_convertedB=times_convertedB[noliveB]
        countrateA = countrateA[noliveA]
        lvtA = lvtA[noliveA]
        countrateB = countrateB[noliveB]
        lvtB = lvtB[noliveB]

        #Remove times with zero counts in the lower energy range (SAA, etc)
        if er == [2.5, 3.5]:
            counttimesA = np.where(countrateA!=0.)[0]
            counttimesB = np.where(countrateA!=0.)[0]

        times_convertedA=times_convertedA[counttimesA]
        times_convertedB=times_convertedB[counttimesB]
        countrateA = countrateA[counttimesA]
        lvtA = lvtA[counttimesA]
        countrateB = countrateB[counttimesB]
        lvtB = lvtB[counttimesB]

        irateA = countrateA*lvtA
        irateB = countrateB*lvtB


        if er==[6.,10.]:
            avgA = np.sum(countrateA*lvtA)/len(lvtA)
            #print(avgA, avgA**(1/2))
            avgB = np.sum(countrateB*lvtB)/len(lvtB)
            #print(avgB, avgB**(1/2))
            glconA, baddiesA = check_qui(countrateA*lvtA, method='global')
            glconB, baddiesB = check_qui(countrateB*lvtB, method='global')
            #print('6-10 keV global consistency: ', glconA, glconB)


            bindex=0
            #If we don't have global consistency in either FPM
            if not np.logical_or(glconA, glconB):
                abs = baddiesA+baddiesB
                print('total baddies:', len(abs), '%: ', len(abs)/(len(lvtA)+len(lvtB)))
                #Mean value of the inconsistent time bins divided by the mean time bin
                #AKA how off-center the inconsistent value distribution is.
                #bindex=np.mean(abs)/np.mean(np.arange(0, len(lvtA)))
                #print(bindex, np.std(baddiesA), len(lvtA))
                for ba in baddiesA:
                    ax2.axvline(times_convertedA[ba], color=ecolors[ind-8]) 
                for bb in baddiesB:
                    ax2.axvline(times_convertedB[bb], color=ecolors[ind-8]) 

            #print(data.keys())
            mntime = data['Mean Time']
            mdtime = data['Median Time']
            #print(type(mntime), type(mdtime))
            ax2.axvline(mntime.datetime64, color='Purple', linestyle='dashed', label='Mean') 
            ax2.axvline(mdtime.datetime64, color='Purple', linestyle='dotted', label='Median') 

            #og method: middle third in time
            #third = (times_convertedA[-1]-times_convertedA[0])/3
            #firstthird = times_convertedA[0]+third
            #secondthird = times_convertedA[0]+2*third
            #inmiddle = firstthird <= mdtime <= secondthird

            #new method: middle third in bins
            third = int(len(times_convertedA)/3)
            #print(third)
            firstthird = times_convertedA[third]
            secondthird = times_convertedA[2*third]
            inmiddle = firstthird <= mdtime <= secondthird

            #print('alltimes', data['All Times'])
            ats = data['All Times']
            ats.sort()
            waittimes = [ats[i+1]-ats[i] for i in range(0, len(ats)-1)]
            #print('wait times', waittimes)
            regular_spacing = (ats[-1]-ats[0])/len(ats)


            #from scipy.special import factorial
            #from scipy.stats import chisquare
            # rate=countrateA*lvtA
            # avgrate = np.sum(rate)/len(rate)
            # ProbPoissA=[(avgrate)**k*np.exp(-avgrate)/factorial(k) for k in rate]
            # chisqA = np.sum([(x-np.mean(rate))**2/np.mean(rate) for x in rate])
            # print('XA: ', chisqA, chisqA/(len(rate)-2))
            # chisqA = chisquare(rate)
            # print('XA: ', chisqA[1], type(chisqA[1]))
            
            # rate=countrateB*lvtB
            # avgrate = np.sum(rate)/len(rate)
            # ProbPoissB=[(avgrate)**k*np.exp(-avgrate)/factorial(k) for k in rate]
            # chisqB = np.sum([(x-np.mean(rate))**2/np.mean(rate) for x in rate])
            # print('XB: ', chisqB, chisqB/(len(rate)-2))
            # chisqB = chisquare(rate)
            # print('XB: ', chisqB[1])

            cash_condition=True
            rate = countrateB*lvtB + countrateA*lvtA
            avgrate = np.sum(rate)/len(rate)
            #chisq = chisquare(rate)
            #c_class = np.sum(cashstatistic.cash_classic(avgrate, rate))
            c_mod = np.sum(cashstatistic.cash_mod(avgrate, rate))
            ec, vc = cashstatistic.cash_mod_expectations(avgrate)
            q=1.3
            N=len(rate)
            c_crit = ec*N + q*(vc*N)**(1/2)
            print('cstats: ', c_mod, c_crit)
            if c_mod > c_crit:
                print('CSTAT over critical value!')
                cash_condition=False
            
        
        # for cc in range(0, min(len(consA), len(consB))):
        #     if np.abs(consA[cc]+consB[cc]) > 1:
        #         ax2.axvline(times_convertedB[cc], color=ecolors[ind-8])    
        #         ax2.axvline(times_convertedA[cc], color=ecolors[ind-8])  
        
        ax2.errorbar(times_convertedA, countrateA*lvtA, yerr=(countrateA*lvtA)**(1/2),
                 label='NuSTAR FPMA Counts '+str(erange[0])+' to '+str(erange[1])+' keV',
                 **default_kwargs, color=clrs[ind])
        ax2.errorbar(times_convertedB, countrateB*lvtB, yerr=(countrateB*lvtB)**(1/2), 
                 label='NuSTAR FPMB Counts '+str(erange[0])+' to '+str(erange[1])+' keV', 
                 **default_kwargs, color=clrs[ind+1])        
        
        ind+=2  


    ax3.plot(times_convertedA, lvtA, 
                 label='NuSTAR FPMA Livetime',
                 **default_kwargs, color='Black')

    #print('times A, B: ', len(times_convertedA), len(times_convertedB)) 
    #print('lvt A, B: ', len(lvtA), len(lvtB))
    ax3.plot(times_convertedB, lvtB, 
                 label='NuSTAR FPMA Livetime',
                 **default_kwargs, color='Red')

    
    if bool(timerange)==False:
        timerange= [ times_convertedA[0], times_convertedA[-1]]

    #print('Using time limits:')
    #print(timerange)

    range3 = [0, 2*np.max(lvtA[np.where(np.logical_and(times_convertedA > timerange[0], times_convertedA < timerange[1]))])]

    ax2.legend(ncol=2, loc='upper center')
    ax2.axhline(avgA, color='green')
    ax2.axhline(avgB, color='green', linestyle='dashed')
    #ax2.set_ylim(range2[0], range2[1])
    ax2.set_xlim(timerange[0], timerange[1])
    ax2.set_title('Lightcurves, Var/Mean: '+str(np.var(rate)/np.mean(rate))+' CSTAT: '+str(c_mod)+' C_Crit: '+str(c_crit)) #, bindex:'+str(bindex))
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    ax2.xaxis.set_minor_locator(mdates.MinuteLocator(interval=1))
    ax2.set_yscale('log')

    ax3.set_ylim(range3[0],range3[1])
    ax3.set_xlim(timerange[0], timerange[1])
    ax3.set_title('Livetimes')
    ax3.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    ax3.xaxis.set_minor_locator(mdates.MinuteLocator(interval=1))
    ax3.legend(loc='upper center')  
    
    #plt.savefig(save_dir+res_save_add+'NuSTAR_lightcurves_err_'+nukeyword+'.png')

    # #fig, ax = plt.subplots(1, 1, figsize=(8, 4))
    # #ax.hist(waittimes, bins=20)
    # #ax.axvline(regular_spacing, color='red')
    # ax4.plot(times_convertedA, ProbPoissA)
    # ax4.plot(times_convertedB, ProbPoissB)
    # #print(ProbPoiss)
    # #print(np.min(ProbPoiss))
    # #print(np.where(ProbPoiss == np.min(ProbPoiss))[0])
    # #print(len(np.where(ProbPoiss == np.min(ProbPoiss))[0]))
    # ProbPoiss = ProbPoissA + ProbPoissB
    # numtimes=len(np.where(ProbPoiss == np.min(ProbPoiss))[0])
    # #print((1.-np.min(ProbPoiss))**len(rate))
    # allsamp = len(lvtA)+len(lvtB)
    # probb = 1.-(1.-np.min(ProbPoiss))**(allsamp-(numtimes-1))
    # #print(probb)
    # #plt.savefig(save_dir+res_save_add+'NuSTAR_lightcurves_poissprob_'+nukeyword+'.png')

    plt.savefig(save_dir+res_save_add+'NuSTAR_lightcurves_err_'+nukeyword+'.png')

    if cash_condition:
        plt.savefig(save_dir+'/new_good_cases2/NuSTAR_lightcurves_err_'+nukeyword+'.png')
    else:
        plt.savefig(save_dir+'/new_bad_cases2/NuSTAR_lightcurves_err_'+nukeyword+'.png')   
        
    return glconA, glconB, inmiddle, cash_condition
    

In [ ]:
erange=[6., 10.]
timebin=10
livetime_corr=True
event_stats=False
save_dir='./quiesence/'
regid=0

import visualize_dem_results as viz
import region_fitting as rf
import lightcurves as lc
import glob

for kk in list(all_targets.keys()):

    print(kk)
    if regid == 2:
        if len(all_targets[kk]['per_region_quiet_time_intervals']) < 3:
            print('1-2 regions')
            continue

    if regid == 1:
        if len(all_targets[kk]['per_region_quiet_time_intervals']) < 2:
            print('1 region')
            continue

        
    qts = all_targets[kk]['per_region_quiet_time_intervals'][regid]

    datapaths = all_targets[kk]['datapaths']
    obsids = all_targets[kk]['obsids']
    #For each orbit for the first region
    for o in range(0, len(qts)):
        contiguous, intervals, int_inds = check_contig(qts[o])
        print(contiguous)
        print(int_inds)
        print('')
        #Load in NuSTAR data and response stuff for the orbit for each FPM
        fpms=['A', 'B']
        evts=[]
        hks=[]
        loaded_nu_stuff = []
        alltimes = []
        for fpm in fpms:
            evt = glob.glob(datapaths[o]+'/event_cl/*'+fpm+'06_cl_sunpos.evt')
            hk  = glob.glob(datapaths[o]+'/hk/*'+fpm+'_fpm.hk')
            evts.append(evt)
            hks.append(hk)
            
            evtdata, hdr = lc.load_nufiles(evt[0])
            lvdata, lvhdr = lc.load_nufiles(hk[0])
            times = nuutil.convert_nustar_time(evtdata['TIME'])
            kev = evtdata['PI']*0.04+1.6
            #We'll need all these later
            loaded_nu_stuff.append([times, evtdata, hdr, kev, lvdata, lvhdr])

        
        #For each of the contiguous quiescent intervals in this orbit
        for ii in intervals:
            ress=[]
            print(ii)
            tr = [ii[0][0], ii[-1][1]]
            timestringi = viz.make_timestring(tr)
            res_files = glob.glob(save_dir+'NuSTAR_lightcurve_'+kk+'_'+timestringi+'_'+str(regid)+'_'+str(erange[0])+'_to_'+str(erange[1])+'_keV.pickle')
            #if regid==0:
            #    res_files = glob.glob(save_dir+'NuSTAR_lightcurve_'+kk+'_'+timestringi+'_'+str(erange[0])+'_to_'+str(erange[1])+'_keV.pickle')
            if len(res_files) > 0:
                print('Results already.')
                continue

            #For each FPM
            for fs in range(0, len(fpms)):
                times, evtdata, hdr, kev, lvdata, lvhdr = loaded_nu_stuff[fs]
                fpm = fpms[fs]
            
                #TIME FILTER
                time_inds = list(np.where(np.logical_and(times > tr[0], times < tr[1]))[0])

                #SPACE FILTER
                #math to get the coordinates into arcseconds from disk center
                coords = [((evtdata['X'][i]-hdr['TCRPX14'])*hdr['TCDLT14'], 
                           (evtdata['Y'][i]-hdr['TCRPX15'])*hdr['TCDLT15']) for i in range(0, len(evtdata['X']))]
                
                if all_targets[kk]['method'] in ['fit', 'input']:
                    regs=[]
                    for ti in ii:
                        timestring = viz.make_timestring(ti)
                        if all_targets[kk]['method'] == 'fit':
                            patht = all_targets[kk]['working_dir']+'/'+timestring+'/'
                            reg_ = patht+'nu'+obsids[o]+fpm+'06_0_4_p_cl_sunpos__region.reg'
                        if all_targets[kk]['method'] == 'input':
                            patht = all_targets[kk]['directories'][regid]+timestring+'/'
                            reg_ = patht+'gauss_cen_'+obsids[o]+'_A_user_input_'+str(regid)+'_'+timestring+'.reg'
                        offset, rad = rf.read_regfile(reg_, ti[0], ti[1], 'hourangle')
                        regs.append(offset)
                    
                    tester_offset = np.mean(np.array(regs), axis=0)

                if all_targets[kk]['method'] == 'double':
                    reg_ = all_targets[kk]['working_dir']+'gauss_cen_'+obsids[o]+'_'+fpm+'_'+str(regid)+'.reg'
                    tester_offset_, rad = rf.read_regfile(reg_, tr[0], tr[1], 'hourangle')
                    tester_offset = np.array(tester_offset_)
                
                circ_inds = [i for i in range(0, len(coords)) if (np.linalg.norm(coords[i] - tester_offset) < 150.)]
            
                    
                #ENERGY FILTER
                erange_inds = list(np.where(np.logical_and(kev > erange[0],kev < erange[1]))[0])

                #Indices satisfying all the conditions
                all_inds = list(set(time_inds).intersection(circ_inds, erange_inds))

                alltimes.extend(evtdata['TIME'][all_inds])
                
                res = lc.get_a_nustar_lightcurve(evtdata[all_inds], hdr, lvdata, lvhdr, 
                                timebin=timebin, livetime_corr=livetime_corr,
                              event_stats=event_stats)


                ress.append(res)
            

            mdt = nuutil.convert_nustar_time(np.median(alltimes))
            mnt = nuutil.convert_nustar_time(np.mean(alltimes))
            
            resA, resB = ress
            data = {'Livetime-Corrected?': livetime_corr,
                'Time Bin (s)': timebin,
                'Energy Range': erange,
                'file paths': evts + hks,
                'FPMA_countrate': resA[1],
                'FPMB_countrate': resB[1],
                'FPMA_counts': resA[3],
                'FPMB_counts': resB[3],
                'FPMA_times': resA[0],
                'FPMB_times': resB[0],
                'FPMA_livetime': resA[2],
                'FPMB_livetime': resB[2],
                'Mean Time': mnt,
                'Median Time': mdt,
                'All Times': alltimes
                }
            
            savefile=save_dir+'NuSTAR_lightcurve_'+kk+'_'+timestringi+'_'+str(regid)+'_'+str(erange[0])+'_to_'+str(erange[1])+'_keV.pickle'
            
            with open(savefile, 'wb') as f:
                # Pickle the 'data' dictionary using the highest protocol available.
                pickle.dump(data, f, pickle.HIGHEST_PROTOCOL)    
        
        print('')
    print('')

In [ ]:
#(if not running the above to make the lightcurve files)
erange=[6., 10.]
timebin=10
livetime_corr=True
event_stats=False
save_dir='./quiesence/'
regid=1

In [ ]:
importlib.reload(lc)
regid=0

bad_intervals=[]
good_intervals=[]
for regid in [0,1]:
    
    for kk in list(all_targets.keys()):
        
        print(kk)
        if regid == 1:
            if len(all_targets[kk]['per_region_quiet_time_intervals']) == 1:
                print('one region')
                continue
        qts = all_targets[kk]['per_region_quiet_time_intervals'][regid]
        #For each orbit for the first region

        for o in range(0, len(qts)):
            contiguous, intervals, int_inds = check_contig(qts[o])
            #print(contiguous)
            #print(int_inds)
            #print('')
            #For each of the contiguous quiescent intervals in this orbit
            for ii in intervals:
                ress=[]
                #print(ii)
                tr = [ii[0][0], ii[-1][1]]
    
                #nukeyword = '_'+kk+'_'+viz.make_timestring(tr)
                #if regid==1:
                nukeyword = '_'+kk+'_'+viz.make_timestring(tr)+'_'+str(regid)

                #print(nukeyword)
                #print(save_dir)
                from datetime import timezone
                trd = [tr[0].datetime, tr[1].datetime]
                trd = [t.replace(tzinfo=timezone.utc) for t in trd]
                print(trd)
                gcA, gcB, inmiddle, pv = plot_nustar_lightcurves_err(save_dir=save_dir, timerange=trd, res_save_add='/all/',
                            eranges = [[2.5,3.5],[3.5,6.],[6.,10.]],
                           nukeyword=nukeyword)

                if not pv:
                    bad_intervals.append(ii)
                else:
                    good_intervals.append(ii)
    
            print('')

In [ ]:
cstat_elim = []

for b in bad_intervals:
    for bb in b:
        cstat_elim.append([bb[0]+5*u.s, bb[1]-5*u.s])


print('bad (contiguous): ', len(bad_intervals), 'bad (individual): ', len(cstat_elim))

cstat_keep = []
allcstattimes = []
for b in good_intervals:
    for bb in b:
        cstat_keep.append([bb[0]+5*u.s, bb[1]-5*u.s])
        allcstattimes.append(bb)


print('good (contiguous): ', len(good_intervals), 'good (individual): ', len(cstat_keep))

print('')
for m in cstat_elim:
    print(m[0].strftime('%D %H-%M-%S'), m[1].strftime('%D %H-%M-%S'))


data = {'fake_cstat_flares': cstat_elim}

# with open('/Users/jmdunca2/do-dem/reference_files/manual_flares_cstat_elim.pickle', 'wb') as f:
#      # Pickle the 'data' dictionary using the highest protocol available.
#      pickle.dump(data, f, pickle.HIGHEST_PROTOCOL) 

In [ ]:
import all_nu_analysis as ana

with open('/Users/jmdunca2/do-dem/reference_files/samesames.pickle', 'rb') as f:
    data = pickle.load(f)

samesames = data['same region lists']

filelist = ana.get_same_region_file_lists(samesames, all_targets)

allfiles=[]
alltimes=[]
for f in filelist:
    f.sort()
    for f_ in f:
        allfiles.append(f_)
        with open(f_, 'rb') as f:
            data = pickle.load(f)
        alltimes.append(data['time_interval'])

The below represents an excruciatingly long winded effort to identify the discrepancy between two accountings of the total number of quiescent DEM intervals. The difference was due to the original accidental exclusion of January 30th 2020 from the "samesames" label list. 

In [ ]:
prqti_all=0
rfd_all=0

allcombos2 = []

for regid in [0,1]:
    for kk in all_targets.keys():
        
        allorbitints = 0
        #PRQTI: first index: region; second index: orbit.
        if len(all_targets[kk]['per_region_quiet_time_intervals']) > regid:
            #Add up the number of times from all orbits:
            for i in range(0, len(all_targets[kk]['per_region_quiet_time_intervals'][regid])):
                allorbitints += len(all_targets[kk]['per_region_quiet_time_intervals'][regid][i])
            #prqti_all += allorbitints
            #rfd_all += len(all_targets[kk]['res_file_dict(s)'][regid]['quiet files all-inst'])
            if allorbitints > 0:
                allcombos2.append(kk+' region_'+str(regid))
                prqti_all += allorbitints
                rfd_all += len(all_targets[kk]['res_file_dict(s)'][regid]['quiet files all-inst'])
    
            #Compare to the number of files in the other list
            if allorbitints == len(all_targets[kk]['res_file_dict(s)'][regid]['quiet files all-inst']):
                #print('good', kk, allorbitints)
                if allorbitints > 0:
                    print('Files for: ', kk, allorbitints)
            else:
                print('AHHHHHHHHHHHHH') 
                print(kk)
                #print(all_targets[kk]['res_file_dict(s)'][1]['quiet files all-inst'])
                #print(all_targets[kk]['per_region_quiet_time_intervals'][1])

print(prqti_all, rfd_all)

In [ ]:
afs=0
AFS_=0

allcombos = []

for i in range(0, len(samesames)):
    print(samesames[i], len(filelist[i]))
    allfiles=0
    for ss in samesames[i]:
        #sub-keys in the key
        #print(ss.split(' ')[0], int(ss.split(' ')[1][-1]))
        allcombos.append(ss)
        sks = all_targets[ss.split(' ')[0]]['per_region_quiet_time_intervals'][int(ss.split(' ')[1][-1])]
        for j in range(0, len(sks)):
            allfiles+=len(sks[j])
            
    if not allfiles == len(filelist[i]):
        print("AAAAAAAAAH")
            
    afs+=len(filelist[i])
print(afs)
print(allcombos)

In [ ]:
print(len(allcombos2), len(allcombos))
culprit = set(allcombos2) - set(allcombos)
culprit

Now, let's get some statistics about the total intervals and regions (including those not ultimately part of the quiescent analysis). 


In [ ]:
quiet_time_regions = []
flare_only_regions = []
other_regions = []
for kk in all_targets.keys():
    for regid in [0,1]:
        if len(all_targets[kk]['res_file_dict(s)']) > regid and any(all_targets[kk]['per_region_all_time_intervals']):
            qfiles = all_targets[kk]['res_file_dict(s)'][regid]['quiet files all-inst']
            ffiles = all_targets[kk]['res_file_dict(s)'][regid]['flare files all-inst']
            
            alltimes=0
            for o in range(0, len(all_targets[kk]['datapaths'])):
                #print(regid, o)
                #print(all_targets[kk]['per_region_all_time_intervals'])
                alltimes += len(all_targets[kk]['per_region_all_time_intervals'][regid][o])

            if len(qfiles) > 0:
                quiet_time_regions.append(kk+' region_'+str(regid))
            elif len(ffiles) > 0: 
                flare_only_regions.append(kk+' region_'+str(regid))
            else:
                other_regions.append(kk+' region_'+str(regid))
                #print(kk, regid)
                #print(len(qfiles), len(ffiles), alltimes)
                #print(all_targets[kk]['per_region_all_time_intervals'][regid])

The quiet time regions have already been catalogued into the samesames list. As 8 of the IDs refer to an already-defined region observed on a second+ day, there are in fact 33 in total. 

In [ ]:
print(len(quiet_time_regions), len(samesames))
samesames

The flare-only regions include the below which have completed flare DEMs but not quiet DEMs. 

- 09-jun-2020 does not count, as it's a flare-only day within a longer observation of the same region on the 6th, 7th, and 8th.
- Additional regions that don't show up here due to earlier edits:
    - 2017-August-21 (great american eclipse) - 1 region
    - 2021-May-7 – 1 region
    - 2022-February-24,25 - 3 regions
    - 2022-February-27 - 1 region
    - 2022-June-3_1 - 1 region
    - 2022-December-10,11_1 - 1 region NEVERMIND, THIS IS THE SAME AS DECEMBER 9.
    - 2023-March-18 - 1 region

In [ ]:
flare_only_regions

Other regions:
- 27-jul-16_1 region_1: no quiet times for the second day observation of the region (see 26-jul-16_1 region_1).
- 29-may-18_2 region_1: FAILS TIS
- 17-nov-21_2 region_0: no quiet times for the first day observation of the region (see 19-nov-21 region_0, etc).
- 24-feb-22: all-flaring (all regions)
- 25-feb-22: all-flaring (all regions - same 2 as 24-feb-22, plus a third new region)
- 27-feb-22: all-flaring (one region)
- 18-mar-23_2: all-flaring (one region)

In [ ]:
other_regions

OTHERS: 

FAILED TIS:
- 2016-April-22 (one region)
- 2021-November-22_2 (one region)


GHOST-RAY specific exclusion:

Ghost-rays with no good extraction region:
- 2015-September-1,2 (two regions)
- 2017-September-11,12,13 (one region)
- 2023-December-28,29 (three regions)

Ghost ray corr breaks DEM:
- 2023-March-18_1 (one region)

In summary:

-  55 total active regions under consideration. 


- 33 regions with quiescent times (included); see samesames list.
    - Of these, 16/33 have ghost ray corrections. 


- 12 regions with only flaring times:
    - 2017-July-16_2 - 1 region
    - 2017-August-21 (great american eclipse) - 1 region
    - 2021-May-7 – 1 region
    - 2022-February-24,25 - 3 regions
    - 2022-February-27 - 1 region
    - 2022-June-3_1 - 1 region
    - 2022-September-6 - 1 region
    - 2022-December-9, 10, 11_1 - 2 regions
    - 2023-March-18 - 1 region

- 3 regions which failed TIS in all observations
    - 2016-April-22 - 1 region
    - 2018-May-29_2 - 1 region
    - 2021-November-22_2 - 1 region

- 7 regions excluded due to ghost ray issues (no background region, or DEM fails after correction)
    - 2015-September-1,2 - 2 regions
    - 2017-September-11,12,13 – 1 region
    - 2023-December-28,29 – 3 regions
    - 2023-March-18_1 - 1 region

In [ ]:
#what is the TOTAL number of time intervals, before the quiescence cut?

quiet_time_count=0
all_time_count=0
biggest_time = 0*u.min
for kk in all_targets.keys():
    #print(kk)
    for reg in range(0, len(all_targets[kk]['per_region_all_time_intervals'])):
        for orb in range(0, len(all_targets[kk]['per_region_all_time_intervals'][reg])):
            #print(reg, orb, ':', len(all_targets[kk]['per_region_all_time_intervals'][reg][orb]))
            #print(reg, orb, ':', len(all_targets[kk]['per_region_quiet_time_intervals'][reg][orb]))
            quiet_time_count += len(all_targets[kk]['per_region_quiet_time_intervals'][reg][orb])
            all_time_count += len(all_targets[kk]['per_region_all_time_intervals'][reg][orb])
            for ti in range(0, len(all_targets[kk]['per_region_all_time_intervals'][reg][orb])):
                time = all_targets[kk]['per_region_all_time_intervals'][reg][orb][ti]
                dur = (time[1]-time[0]).to(u.min)
                if dur > biggest_time:
                    biggest_time = dur


print(biggest_time)



print(len(list(all_targets.keys())))
print(all_time_count)
print(quiet_time_count)


additional_rejected_regions=['01-sep-15', '02-sep-15', '21-aug-17', '11-sep-17', '12-sep-17', '13-sep-17', '07-may-21', '28-dec-23']